<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-07-25

| Package | Version |
|---------|---------|
| **nnsight** | **0.8** |
| Python | 3.12.13 |
| torch | 2.13.0+cu126 |
| transformers | 5.15.0 |

</details>


# Skipping Modules

Use `.skip()` to bypass a module's forward pass entirely, substituting a value you provide as its output. When the model is about to run the module, it won't — your `replacement` is used instead, and none of the module's inner computation happens. This lets you ablate a component, route around a layer, or inject an externally-computed activation (e.g. a reconstruction from an SAE) at a specific point.

## Setup

`TransformersModel` is the primary class for HuggingFace models in nnsight 0.8. Pass a repo id; `dispatch=True` loads the weights right away instead of on the first trace.

In [1]:
from nnsight.modeling.transformers import TransformersModel

model = TransformersModel("openai-community/gpt2", device_map="auto", dispatch=True)

/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Skipping a Module

Call `.skip(replacement)` on any module to bypass its forward pass. The `replacement` is used as the module's output instead. Here we skip layer 5's MLP and feed the previous layer's MLP output through in its place — the MLP's own forward never runs.

In [2]:
# Normal prediction
with model.trace("The Eiffel Tower is in the city of"):
    normal_logits = model.lm_head.output.save()

# Skip layer 5's MLP — use the previous MLP's output as its output instead
with model.trace("The Eiffel Tower is in the city of"):
    model.transformer.h[5].mlp.skip(model.transformer.h[4].mlp.output)
    skipped_logits = model.lm_head.output.save()

print(f"Normal:  {model.tokenizer.decode(normal_logits[0, -1].argmax(dim=-1))}")
print(f"Skipped: {model.tokenizer.decode(skipped_logits[0, -1].argmax(dim=-1))}")

Normal:   Paris
Skipped:  Paris


## Pass-through with a Module's Own Input

Skipping a module with its own `.input` turns it into a pass-through: the input is handed straight to the output and the module's forward is bypassed. For a transformer block this is a clean way to drop the block's contribution entirely.

In [3]:
with model.trace("The Eiffel Tower is in the city of"):
    # Layer 6 becomes a no-op: its input passes straight through
    model.transformer.h[6].skip(model.transformer.h[6].input)
    passthrough_logits = model.lm_head.output.save()

print(f"Layer 6 passed through: {model.tokenizer.decode(passthrough_logits[0, -1].argmax(dim=-1))}")

Layer 6 passed through:  London


## Skipping Multiple Layers

You can skip a range of layers in a loop. Pass the output of the last non-skipped layer as the replacement for each skipped layer.

In [4]:
with model.trace("The Eiffel Tower is in the city of"):
    # Skip layers 3 through 8, reusing layer 2's output for each
    replacement = model.transformer.h[2].output
    for i in range(3, 9):
        model.transformer.h[i].skip(replacement)

    logits = model.lm_head.output.save()

print(f"Skipped layers 3-8: {model.tokenizer.decode(logits[0, -1].argmax(dim=-1))}")

Skipped layers 3-8:  the


<details class="admonition note">
<summary>Replacement shape and type must match the module's real output</summary>

The `replacement` must match what the module would normally return, or the model's forward will error downstream. What a module returns is model- and `transformers`-version-dependent: some modules hand back a plain tensor `(batch, seq, hidden)`, others return a tuple (or another structure) — so the replacement you pass has to match whatever *that* module actually returns for *your* model and version. When you feed one module's `.output` to another's `.skip()`, it drops in cleanly precisely because both share that same return type. When unsure, read the module's `.output` first and check its type and shape with `print`/`type`/`.shape` before choosing a replacement.

</details>

## Measuring Layer Importance

Skip each layer one at a time and check how the prediction changes — a simple way to measure which layers matter most for a given prompt.

In [5]:
prompt = "The Eiffel Tower is in the city of"

with model.trace(prompt):
    baseline = model.lm_head.output[0, -1].argmax(dim=-1).save()

baseline_token = model.tokenizer.decode(baseline)
print(f"Baseline: {baseline_token}\n")

for layer_idx in range(1, 12):
    with model.trace(prompt):
        model.transformer.h[layer_idx].skip(model.transformer.h[layer_idx - 1].output)
        pred = model.lm_head.output[0, -1].argmax(dim=-1).save()

    token = model.tokenizer.decode(pred)
    changed = " <- changed!" if token != baseline_token else ""
    print(f"Skip layer {layer_idx:2d}: {token}{changed}")

Baseline:  Paris

Skip layer  1:  London <- changed!
Skip layer  2:  London <- changed!
Skip layer  3:  the <- changed!
Skip layer  4:  Paris
Skip layer  5:  Paris
Skip layer  6:  London <- changed!
Skip layer  7:  Paris
Skip layer  8:  London <- changed!
Skip layer  9:  London <- changed!
Skip layer 10:  London <- changed!
Skip layer 11:  Paris


<details class="admonition warning">
<summary>Gotchas</summary>

- **A skipped module's inner ops are unreachable.** Since the forward never runs, requesting a skipped module's sub-modules or `.source` operations raises an out-of-order error — they never execute.
- **`skip` only works inside an active trace**, and a skip is one-shot per module call. During multi-token generation, each step needs its own skip (see `tracer.iter[...]`).
- **Across batched invokes, a `.skip()` must cover every row.** A shared forward can't run for only some rows — skip the module in every invoke or none, and each invoke's replacement fills its own rows.
- **Use `tracer.stop()` to abort the whole forward** — `skip` only bypasses one module.

</details>

<details class="admonition tip">
<summary>When to use skip</summary>

- **Ablation studies** — measure the causal effect of removing a layer or sub-module
- **Layer importance** — identify which layers are critical for specific predictions
- **Model splicing** — replace a module's computation with an alternative output (e.g. an SAE reconstruction)

</details>